# GeoClean bootstrap detector

Trains a small YOLOv8n detector on 1,019 already-boxed images (Roboflow overflow + piles sets) with two classes, `roadside_dump` and `overflowing_bin`, then runs it on the 707 QR4Change dump photos, which have no boxes at all.

**This is a bootstrap model, not the final detector.** Its only job is to pre-label QR4Change so a person corrects boxes in Label Studio instead of drawing every one from scratch.

**Before running:** *Runtime -> Change runtime type -> T4 GPU*. Then *Runtime -> Run all*, and upload the two files from `data/bootstrap/zips/` on your computer when each cell asks: `bootstrap_train.zip` (74 MB) first, `qr4_infer.zip` (269 MB) second.

In [ ]:
!pip install -q ultralytics

## 1. Upload and extract the training set
Select `bootstrap_train.zip` when the picker opens.

In [ ]:
from google.colab import files
import zipfile, os, shutil

def extract(zip_name, dest):
    # Windows zip tools can store paths as images\train\x.jpg; on Linux that becomes one
    # oddly named file instead of folders, so normalise separators while extracting.
    shutil.rmtree(dest, ignore_errors=True)
    with zipfile.ZipFile(zip_name) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/')
            if name.endswith('/'):
                continue
            target = os.path.join(dest, name)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as out:
                shutil.copyfileobj(src, out)

uploaded = files.upload()
zip_name = next(iter(uploaded))
extract(zip_name, '/content/dataset')
for split in ('train', 'val'):
    folder = f'/content/dataset/images/{split}'
    if not os.path.isdir(folder):
        raise SystemExit(f'{folder} is missing. Did you upload bootstrap_train.zip (about 74 MB) here? You uploaded {zip_name}.')
    print(split, len(os.listdir(folder)), 'images')

## 2. Train
Takes roughly 15-25 minutes on a T4. `patience=15` stops early if validation loss plateaus.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    seed=20260913,
    project='/content/runs',
    name='bootstrap',
)

## 3. Check validation metrics
This bootstrap model was trained on the same source class definitions as the pre-filled Label Studio boxes, so mAP here is a sanity check, not a real-world figure. Domain gap to Pune scenes and QR4Change specifically has not been measured.

In [ ]:
metrics = model.val()
print(metrics.box.map50, 'mAP50')
print('per-class mAP50:', dict(zip(['roadside_dump', 'overflowing_bin'], metrics.box.maps)))

## 4. Upload the QR4Change images and run inference
Select `qr4_infer.zip` when the picker opens. This is the step that produces starting boxes for the unlabelled set.

In [ ]:
uploaded = files.upload()
zip_name = next(iter(uploaded))
extract(zip_name, '/content/qr4_infer')
images = sorted(f for f in os.listdir('/content/qr4_infer') if f.lower().endswith('.jpg') and f.startswith('qr4-'))
if not images:
    raise SystemExit(f'No qr4-*.jpg images found. Did you upload qr4_infer.zip (about 269 MB) here? You uploaded {zip_name}.')
print(len(images), 'images')

In [ ]:
import glob, json

# Ultralytics saves to bootstrap2, bootstrap3... when an earlier run folder exists, so use
# the newest best.pt rather than a fixed path.
candidates = glob.glob('/content/runs/**/weights/best.pt', recursive=True)
if not candidates:
    raise SystemExit('No trained best.pt found under /content/runs. Did the training cell finish?')
BEST = max(candidates, key=os.path.getmtime)
RUN_DIR = os.path.dirname(os.path.dirname(BEST))
print('using', BEST)

best = YOLO(BEST)
CLASSES = ['roadside_dump', 'overflowing_bin']
predictions = []
no_detection = 0

for name in images:
    item_id = os.path.splitext(name)[0]
    result = best.predict(f'/content/qr4_infer/{name}', conf=0.25, verbose=False)[0]
    boxes = []
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxyn[0].tolist()  # normalised 0..1: independent of the resize done before upload
        boxes.append({
            'class': CLASSES[int(box.cls[0])],
            'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
            'confidence': round(float(box.conf[0]), 3),
        })
    if not boxes:
        no_detection += 1
    predictions.append({'item_id': item_id, 'boxes': boxes})

with open('/content/qr4_predictions.json', 'w') as f:
    json.dump({'classes': CLASSES, 'model': 'yolov8n-bootstrap', 'predictions': predictions}, f, indent=1)

print(f'{len(images) - no_detection} of {len(images)} images got at least one box')

## 5. Download the results
Downloads one zip with the predictions JSON, the trained weights and the training curves. Save it as `data/bootstrap/qr4_predictions_result.zip` in the project folder, back on your computer.

In [ ]:
import glob, shutil, zipfile

with zipfile.ZipFile('/content/geoclean_bootstrap_results.zip', 'w') as z:
    z.write('/content/qr4_predictions.json', 'qr4_predictions.json')
    z.write(BEST, 'bootstrap_best.pt')
    for p in glob.glob(f'{RUN_DIR}/*.png') + glob.glob(f'{RUN_DIR}/results.csv'):
        z.write(p, os.path.basename(p))

files.download('/content/geoclean_bootstrap_results.zip')